In [1]:
!pip3 install gurobipy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 47.5 MB/s eta 0:00:00 0:00:01

[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python3.13 -m pip install --upgrade pip


In [5]:
from gurobipy import *
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
import numpy as np

In [20]:
df = pd.read_csv("w5000.csv", index_col=0)
# If an asset is missing any data, exclude
df = df.dropna(axis=1)
df

,w5000,A.Close,AA.Close,AAL.Close,AAME.Close,AAN.Close,AAON.Close,AAP.Close,AAPL.Close,AAWW.Close,...,YUMA.Close,ZAZA.Close,ZBH.Close,ZBRA.Close,ZEUS.Close,ZION.Close,ZIOP.Close,ZIXI.Close,ZN.Close,ZUMZ.Close
1,14246.71000,24.535049,70.479988,56.299999,3.05,17.000000,5.260247,35.580002,11.971429,44.000000,...,68.639999,257.700012,77.099998,34.880001,21.650000,82.910004,5.97,1.21,11.72,31.049999
2,14269.90000,24.613733,69.951332,58.840000,3.11,17.166666,5.325432,35.810001,12.237143,45.279999,...,65.599998,258.600006,78.820000,34.680000,21.990000,83.279999,5.78,1.17,10.40,32.439999
3,14164.80000,24.384836,69.110283,58.290001,3.42,16.913334,5.185185,35.020000,12.150000,45.160000,...,65.279999,257.899994,78.769997,34.330002,21.740000,83.029999,5.94,1.15,12.50,32.570000
4,14197.15000,24.298998,68.437439,57.930000,3.59,17.133333,5.234568,35.139999,12.210000,45.610001,...,66.400002,250.800003,78.260002,34.400002,21.540001,83.169998,5.82,1.19,12.03,32.950001
5,14204.71000,24.327610,68.533562,57.900002,3.63,17.006666,5.250371,35.439999,13.224286,46.060001,...,66.400002,247.600006,78.309998,34.340000,21.559999,83.629997,5.70,1.23,11.50,33.939999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2765,27864.83984,67.349998,49.990002,52.590000,3.85,40.259998,36.750000,100.550003,175.009995,60.150002,...,1.130000,0.030000,120.120003,105.150002,22.150000,51.330002,4.05,4.45,2.31,21.450001
2766,27850.16992,67.250000,50.380001,52.849998,3.60,40.360001,36.599998,101.959999,170.570007,59.849998,...,1.070000,0.030000,119.959999,104.790001,22.200001,50.860001,4.07,4.44,2.36,21.850000
2767,27864.58984,67.300003,51.840000,52.400002,3.35,40.599998,36.500000,99.769997,170.600006,59.400002,...,1.130000,0.030000,120.139999,104.949997,21.830000,50.709999,4.03,4.56,2.38,21.150000
2768,27920.94922,67.449997,54.139999,52.459999,3.25,40.240002,36.799999,99.709999,171.080002,59.150002,...,1.150000,0.030000,121.750000,104.279999,21.700001,51.340000,4.00,4.41,2.40,21.200001


In [22]:
# calculate the return values
returns = df.pct_change().dropna()
returns

,w5000,A.Close,AA.Close,AAL.Close,AAME.Close,AAN.Close,AAON.Close,AAP.Close,AAPL.Close,AAWW.Close,...,YUMA.Close,ZAZA.Close,ZBH.Close,ZBRA.Close,ZEUS.Close,ZION.Close,ZIOP.Close,ZIXI.Close,ZN.Close,ZUMZ.Close
2,0.001628,0.003207,-0.007501,0.045115,0.019672,0.009804,0.012392,0.006464,0.022196,0.029091,...,-0.044289,0.003492,0.022309,-0.005734,0.015704,0.004463,-0.031826,-0.033058,-0.112628,0.044767
3,-0.007365,-0.009300,-0.012023,-0.009347,0.099678,-0.014757,-0.026335,-0.022061,-0.007121,-0.002650,...,-0.004878,-0.002707,-0.000634,-0.010092,-0.011369,-0.003002,0.027682,-0.017094,0.201923,0.004007
4,0.002284,-0.003520,-0.009736,-0.006176,0.049708,0.013007,0.009524,0.003427,0.004938,0.009965,...,0.017157,-0.027530,-0.006474,0.002039,-0.009200,0.001686,-0.020202,0.034783,-0.037600,0.011667
5,0.000533,0.001177,0.001405,-0.000518,0.011142,-0.007393,0.003019,0.008537,0.083070,0.009866,...,0.000000,-0.012759,0.000639,-0.001744,0.000928,0.005531,-0.020619,0.033613,-0.044057,0.030045
6,0.002664,-0.009115,0.059958,0.017789,0.022039,0.000000,-0.002634,0.001411,0.047856,-0.001520,...,-0.036145,-0.019386,-0.003320,0.000874,0.012059,0.005859,0.000000,0.105691,-0.152174,0.039776
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2765,-0.000425,-0.002518,0.020412,-0.003789,0.000000,-0.007152,0.006849,0.004195,0.000000,0.005013,...,-0.025862,0.000000,0.001417,-0.012954,-0.016430,-0.002526,-0.026442,0.009070,-0.029412,-0.002326
2766,-0.000526,-0.001485,0.007802,0.004944,-0.064935,0.002484,-0.004082,0.014023,-0.025370,-0.004988,...,-0.053097,0.000000,-0.001332,-0.003424,0.002257,-0.009156,0.004938,-0.002247,0.021645,0.018648
2767,0.000518,0.000744,0.028980,-0.008515,-0.069444,0.005946,-0.002732,-0.021479,0.000176,-0.007519,...,0.056075,0.000000,0.001501,0.001527,-0.016667,-0.002949,-0.009828,0.027027,0.008475,-0.032037
2768,0.002023,0.002229,0.044367,0.001145,-0.029851,-0.008867,0.008219,-0.000601,0.002814,-0.004209,...,0.017699,0.000000,0.013401,-0.006384,-0.005955,0.012424,-0.007444,-0.032895,0.008403,0.002364


If, for an asset, the return for any given day exceeds 0.25 in absolute value, replace the return for that day
with a moving average (of returns). If, for an asset, you need to perform this fix more than twenty times,
exclude that asset.

In [23]:
def to_average(col):
    col_fixed = col.copy()
    num_fixed = 0
    for i in range(len(col)):
        # if the returns exceed 0.25, replace it with a moving average
        if abs(col.iloc[i]) > 0.25:
            if i >= 2 and i + 2 < len(col):
                window = col.iloc[i-2:i+3]
                average = window[window.abs() <= 0.25].mean()
                col_fixed.iloc[i] = average
                num_fixed += 1
    return col_fixed, num_fixed


In [30]:
cleaned_data = {}
for col in returns.columns:
    fixed_col, num_changes = to_average(returns[col])
    if num_changes <= 20:
        cleaned_data[col] = fixed_col

df_final = pd.DataFrame(cleaned_data)
# number of assets
df_final.shape[1]


2035

Construct the vector of average asset returns for this list of assets, as well as the covariance matrix of returns.

In [28]:
mu = df_final.mean()
cov_matrix = df_final.cov()
